# Hypothesis Testing Practice Notebook

This notebook is a hands-on companion to the Markdown file on **Hypothesis Testing**.  
It includes practical examples for both **parametric** and **nonparametric** tests.

Topics covered:

## Parametric tests
1. z-test  
2. one-sample t-test  
3. independent t-test  
4. paired t-test  
5. one-way ANOVA  
6. repeated-measures style comparison  

## Nonparametric tests
7. Mann–Whitney U test  
8. Wilcoxon signed-rank test  
9. Kruskal–Wallis test  
10. Friedman test  
11. Sign test  
12. Chi-square test  

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statistics import NormalDist
from math import comb
from scipy import stats

np.random.seed(42)

## Create Example Datasets

We generate several small example datasets that will be reused throughout the notebook.

In [ ]:
sample_one = np.random.normal(loc=52, scale=8, size=30)
group_a = np.random.normal(loc=100, scale=12, size=35)
group_b = np.random.normal(loc=94, scale=11, size=35)

before = np.random.normal(loc=80, scale=10, size=20)
after = before - np.random.normal(loc=4, scale=3, size=20)

anova_g1 = np.random.normal(loc=20, scale=4, size=25)
anova_g2 = np.random.normal(loc=24, scale=4, size=25)
anova_g3 = np.random.normal(loc=28, scale=4, size=25)

benchmark_df = pd.DataFrame({
    'Method_A': [12.1, 11.8, 12.4, 11.9, 12.0, 11.7],
    'Method_B': [11.7, 11.3, 11.9, 11.5, 11.6, 11.2],
    'Method_C': [12.5, 12.0, 12.7, 12.1, 12.3, 12.0],
    'Method_D': [11.9, 11.5, 12.1, 11.6, 11.8, 11.4]
}, index=['Problem_1', 'Problem_2', 'Problem_3', 'Problem_4', 'Problem_5', 'Problem_6'])

benchmark_df

# Part I — Parametric Tests

## 1. z-Test

A z-test is used to test a population mean when the population standard deviation is known, or when the sample is large enough for a normal approximation.

Test statistic:

$$
z = \frac{\bar{x} - \mu_0}{\sigma/\sqrt{n}}
$$

Here we assume a known population standard deviation.

In [ ]:
mu_0 = 50
sigma_known = 8
xbar = sample_one.mean()
n = len(sample_one)

z_stat = (xbar - mu_0) / (sigma_known / np.sqrt(n))
p_value_z = 2 * (1 - NormalDist().cdf(abs(z_stat)))

pd.DataFrame({'Statistic': ['Sample Mean', 'z Statistic', 'p Value'], 'Value': [xbar, z_stat, p_value_z]})

## 2. One-Sample t-Test

Used when the population standard deviation is unknown.

Test statistic:

$$
t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}}
$$

In [ ]:
t_stat_1samp, p_value_1samp = stats.ttest_1samp(sample_one, popmean=50)
pd.DataFrame({'Statistic': ['t Statistic', 'p Value'], 'Value': [t_stat_1samp, p_value_1samp]})

## 3. Independent t-Test

Used to compare two independent group means.

A common Welch-type form is:

$$
t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}
$$

In [ ]:
t_stat_ind, p_value_ind = stats.ttest_ind(group_a, group_b, equal_var=False)

pd.DataFrame({
    'Group': ['A', 'B'],
    'Mean': [group_a.mean(), group_b.mean()],
    'Std': [group_a.std(ddof=1), group_b.std(ddof=1)],
    'n': [len(group_a), len(group_b)]
}), pd.DataFrame({'Statistic': ['t Statistic', 'p Value'], 'Value': [t_stat_ind, p_value_ind]})

## 4. Paired t-Test

Used when two measurements are paired, such as before-vs-after measurements on the same subjects.

If \(d_i = x_i - y_i\), then:

$$
t = \frac{\bar{d}}{s_d/\sqrt{n}}
$$

In [ ]:
t_stat_paired, p_value_paired = stats.ttest_rel(before, after)

paired_summary = pd.DataFrame({
    'Before Mean': [before.mean()],
    'After Mean': [after.mean()],
    'Mean Difference': [(before - after).mean()],
    't Statistic': [t_stat_paired],
    'p Value': [p_value_paired]
})
paired_summary

## 5. One-Way ANOVA

ANOVA tests whether three or more group means are equal.

The F statistic is:

$$
F = \frac{MS_{between}}{MS_{within}}
$$

In [ ]:
f_stat, p_value_anova = stats.f_oneway(anova_g1, anova_g2, anova_g3)

anova_summary = pd.DataFrame({
    'Group': ['G1', 'G2', 'G3'],
    'Mean': [anova_g1.mean(), anova_g2.mean(), anova_g3.mean()],
    'Std': [anova_g1.std(ddof=1), anova_g2.std(ddof=1), anova_g3.std(ddof=1)]
})
anova_summary, pd.DataFrame({'Statistic': ['F Statistic', 'p Value'], 'Value': [f_stat, p_value_anova]})

## 6. Repeated-Measures Style Comparison

A full repeated-measures ANOVA usually requires a dedicated modeling framework.  
Here we create a matched benchmark-style dataset where the same problems are evaluated by several methods.

This setting is often better analyzed with the **Friedman test** in nonparametric benchmarking.

In [ ]:
benchmark_df

# Part II — Nonparametric Tests

## 7. Mann–Whitney U Test

This is the nonparametric alternative to the independent t-test.  
It is used for two independent groups when normality is doubtful.

In [ ]:
u_stat, p_value_mw = stats.mannwhitneyu(group_a, group_b, alternative='two-sided')
pd.DataFrame({'Statistic': ['U Statistic', 'p Value'], 'Value': [u_stat, p_value_mw]})

## 8. Wilcoxon Signed-Rank Test

This is the nonparametric alternative to the paired t-test.  
It is very important for comparing two methods on the same datasets or problem instances.

In [ ]:
method_a = benchmark_df['Method_A']
method_b = benchmark_df['Method_B']

w_stat, p_value_wil = stats.wilcoxon(method_a, method_b, alternative='two-sided')
pd.DataFrame({'Statistic': ['Wilcoxon Statistic', 'p Value'], 'Value': [w_stat, p_value_wil]})

## 9. Kruskal–Wallis Test

This is the nonparametric alternative to one-way ANOVA.  
It is used for comparing three or more independent groups.

In [ ]:
h_stat, p_value_kw = stats.kruskal(anova_g1, anova_g2, anova_g3)
pd.DataFrame({'Statistic': ['H Statistic', 'p Value'], 'Value': [h_stat, p_value_kw]})

## 10. Friedman Test

The Friedman test compares three or more matched methods across the same problems.  
This is one of the most important tests in optimization and benchmark-comparison papers.

In [ ]:
friedman_stat, p_value_friedman = stats.friedmanchisquare(
    benchmark_df['Method_A'],
    benchmark_df['Method_B'],
    benchmark_df['Method_C'],
    benchmark_df['Method_D']
)

pd.DataFrame({'Statistic': ['Friedman Statistic', 'p Value'], 'Value': [friedman_stat, p_value_friedman]})

### Average Ranks for the Benchmark Methods

Lower values are assumed better here, so lower result gets better rank.

In [ ]:
ranks = benchmark_df.rank(axis=1, method='average', ascending=True)
average_ranks = ranks.mean(axis=0).sort_values()
average_ranks

## 11. Sign Test

The sign test is a simple paired nonparametric test that uses only the sign of the differences.

Under the null hypothesis, the number of positive signs follows:

$$
X \sim \text{Binomial}(n, 0.5)
$$

In [ ]:
diff = method_a - method_b
positive = np.sum(diff > 0)
negative = np.sum(diff < 0)
n_eff = positive + negative

def binom_two_sided_p(k, n, p=0.5):
    probs = [comb(n, i) * (p ** i) * ((1 - p) ** (n - i)) for i in range(n + 1)]
    observed = probs[k]
    return sum(pr for pr in probs if pr <= observed + 1e-12)

k = min(positive, negative)
p_value_sign = binom_two_sided_p(k, n_eff, 0.5)

pd.DataFrame({
    'Positive Signs': [positive],
    'Negative Signs': [negative],
    'Effective n': [n_eff],
    'Approx Two-Sided p Value': [p_value_sign]
})

## 12. Chi-Square Test

The chi-square test is used for categorical frequency data.

### 12.1 Chi-Square Test of Independence

We create a contingency table and test whether two categorical variables are associated.

In [ ]:
contingency = pd.DataFrame({
    'Low': [18, 10, 7],
    'Medium': [12, 14, 9],
    'High': [5, 11, 14]
}, index=['Type_A', 'Type_B', 'Type_C'])

chi2_stat, p_value_chi2, dof, expected = stats.chi2_contingency(contingency)

contingency, pd.DataFrame({'Statistic': ['Chi-Square', 'Degrees of Freedom', 'p Value'], 'Value': [chi2_stat, dof, p_value_chi2]})

### Expected Counts

For a chi-square independence test, the expected count in each cell is:

$$
E_{ij} = \frac{(\text{row total})(\text{column total})}{\text{grand total}}
$$

In [ ]:
expected_df = pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)
expected_df

## 13. Small Comparison Summary Table

This table gathers the main test results from the notebook.

In [ ]:
summary = pd.DataFrame({
    'Test': [
        'z-test',
        'one-sample t-test',
        'independent t-test',
        'paired t-test',
        'one-way ANOVA',
        'Mann–Whitney U',
        'Wilcoxon signed-rank',
        'Kruskal–Wallis',
        'Friedman test',
        'Sign test',
        'Chi-square test'
    ],
    'Statistic': [
        z_stat,
        t_stat_1samp,
        t_stat_ind,
        t_stat_paired,
        f_stat,
        u_stat,
        w_stat,
        h_stat,
        friedman_stat,
        positive,
        chi2_stat
    ],
    'p Value': [
        p_value_z,
        p_value_1samp,
        p_value_ind,
        p_value_paired,
        p_value_anova,
        p_value_mw,
        p_value_wil,
        p_value_kw,
        p_value_friedman,
        p_value_sign,
        p_value_chi2
    ]
})
summary

## 14. Mini Exercises

Try these on your own:

1. Change the null mean in the z-test and one-sample t-test.  
2. Generate two groups with closer means and compare the independent t-test and Mann–Whitney U results.  
3. Create a stronger paired effect and compare the paired t-test with Wilcoxon.  
4. Add a fifth method to the benchmark table and rerun the Friedman test.  
5. Modify the contingency table and see how the chi-square p-value changes.  
6. For optimization-style benchmarking, replace the dummy values in `benchmark_df` with your own algorithm results.

These exercises are especially useful for AI, machine learning, structural engineering, and optimization research.